# 02 — Estatística descritiva

Produz painéis exatos anuais, mensais e por dimensões substantivas.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==2.0.2",
        "pandas==2.2.3",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
ABI_CHECK = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import numpy as np; import pandas as pd; "
            "assert np.__version__ == '2.0.2', np.__version__; "
            "assert pd.__version__ == '2.2.3', pd.__version__; "
            "print(f'NumPy {np.__version__}; pandas {pd.__version__}')"
        ),
    ],
    check=True,
    text=True,
    capture_output=True,
)
import numpy as np
import pandas as pd

assert np.__version__ == "2.0.2", f"Reinicie a sessao do Colab: NumPy carregado={np.__version__}"
assert pd.__version__ == "2.2.3", f"Reinicie a sessao do Colab: pandas carregado={pd.__version__}"
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())
print("ABI:", ABI_CHECK.stdout.strip())

## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [ ]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260713-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = False

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

## Decisão metodológica

Bootstrap não é automático: use-o apenas para uma pergunta que declare explicitamente sua população de generalização.

In [ ]:
DESCRITIVAS_SNAPSHOT_PATH = RUN_OUTPUT_ROOT / "00_snapshot" / "discursos_plenario_snapshot.parquet"
assert DESCRITIVAS_SNAPSHOT_PATH.exists(), "Execute o caderno 00."

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [ ]:
from analise.discursos_plenario.descritivas import run_descriptives

DESCRITIVAS_RESULT = None
if RODAR_ETAPA:
    DESCRITIVAS_RESULT = run_descriptives(data_root=DATA_ROOT, run_id=RUN_ID, config_path=CONFIG_PATH)
    print(DESCRITIVAS_RESULT["manifest_path"])
else:
    print("Descritivas não executadas.")

## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [ ]:
import pandas as pd

DESCRITIVAS_ANUAL_PATH = RUN_OUTPUT_ROOT / "02_descritivas" / "anual.csv"
if DESCRITIVAS_ANUAL_PATH.exists():
    DESCRITIVAS_ANUAL = pd.read_csv(DESCRITIVAS_ANUAL_PATH)
    assert DESCRITIVAS_ANUAL["discursos"].ge(0).all()
    display(DESCRITIVAS_ANUAL.tail(12))